In [2]:
pip install requests beautifulsoup4


  Using cached beautifulsoup4-4.14.2-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached beautifulsoup4-4.14.2-py3-none-any.whl (106 kB)
Using cached soupsieve-2.8-py3-none-any.whl (36 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import time

# === DB準備 ===
conn = sqlite3.connect("google_repos.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS repositories (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    language TEXT,
    stars INTEGER
)
""")
conn.commit()

# === スクレイピング対象URL ===
base_url = "https://github.com/google?page={}&tab=repositories"

# 最大ページ数（必要に応じて増やせる）
max_pages = 3

for page in range(1, max_pages + 1):
    print(f"ページ {page} を取得中...")

    try:
        url = base_url.format(page)
        response = requests.get(url)
        response.raise_for_status()

    except requests.exceptions.HTTPError as e:
        print("HTTPエラー:", e)
        continue
    except requests.exceptions.RequestException as e:
        print("通信エラー:", e)
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    repos = soup.select("li source.repo-list-item")

    # 新UIの場合のCSS変更に対応
    repos = soup.select("li.Box-row")

    for repo in repos:
        # リポジトリ名
        name_tag = repo.select_one("a[href*='/google/']")
        if not name_tag:
            continue

        name = name_tag.text.strip()

        # 言語（ない場合もある）
        lang_tag = repo.select_one("span[itemprop='programmingLanguage']")
        language = lang_tag.text.strip() if lang_tag else "N/A"

        # スター数
        star_tag = repo.select_one("a[href$='/stargazers']")
        if star_tag:
            stars = star_tag.text.strip().replace(",", "")
        else:
            stars = "0"

        try:
            stars = int(stars)
        except ValueError:
            stars = 0

        print(f"- {name} / {language} / ⭐ {stars}")

        # DBに保存
        cur.execute(
            "INSERT INTO repositories (name, language, stars) VALUES (?, ?, ?)",
            (name, language, stars)
        )
        conn.commit()

        # Githubに負荷をかけないため
        time.sleep(1)

print("\n=== 保存データ一覧（SELECT） ===")

for row in cur.execute("SELECT * FROM repositories"):
    print(row)

conn.close()


ページ 1 を取得中...
- zerocopy / Rust / ⭐ 2081
- xls / C++ / ⭐ 1375
- ksp / Kotlin / ⭐ 3307
- adk-go / Go / ⭐ 4908
- site-kit-wp / JavaScript / ⭐ 1338
- wasefire / Rust / ⭐ 129
- orbax / Python / ⭐ 455
- adk-python / Python / ⭐ 15612
- osv-scalibr / Go / ⭐ 538
- or-tools / C++ / ⭐ 12715
ページ 2 を取得中...
- zerocopy / Rust / ⭐ 2081
- xls / C++ / ⭐ 1375
- ksp / Kotlin / ⭐ 3307
- adk-go / Go / ⭐ 4908
- site-kit-wp / JavaScript / ⭐ 1338
- wasefire / Rust / ⭐ 129
- orbax / Python / ⭐ 455
- adk-python / Python / ⭐ 15612
- osv-scalibr / Go / ⭐ 538
- or-tools / C++ / ⭐ 12715
ページ 3 を取得中...
- zerocopy / Rust / ⭐ 2081
- xls / C++ / ⭐ 1375
- ksp / Kotlin / ⭐ 3307
- adk-go / Go / ⭐ 4908
- site-kit-wp / JavaScript / ⭐ 1338
- wasefire / Rust / ⭐ 129
- orbax / Python / ⭐ 455
- adk-python / Python / ⭐ 15612
- osv-scalibr / Go / ⭐ 538
- or-tools / C++ / ⭐ 12715

=== 保存データ一覧（SELECT） ===
(1, 'zerocopy', 'Rust', 2081)
(2, 'xls', 'C++', 1375)
(3, 'ksp', 'Kotlin', 3307)
(4, 'adk-go', 'Go', 4908)
(5, 'site-kit-wp', 'Ja